# Alphabet Recognition - PyTorch Training
Train a CNN model using PyTorch on Google Colab

## Step 1: Install dependencies

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
from bidict import bidict
from sklearn.utils import shuffle
import os

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Step 2: Upload your data files

In [ ]:
from google.colab import files

# Create data directory
os.makedirs('data', exist_ok=True)

print("Upload your data files:")
print("1. images.npy")
print("2. labels.npy")
print()

uploaded = files.upload()

# Move files to data directory
for filename in uploaded.keys():
    if filename.endswith('.npy'):
        os.rename(filename, f'data/{filename}')
        print(f"✓ Moved {filename} to data/")

## Step 3: Configure and load data

In [ ]:
# Configuration
BATCH_SIZE = 16
EPOCHS = 20
LEARNING_RATE = 0.001
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Encoder
ENCODER = bidict({
    'A': 1, 'B': 2, 'C': 3, 'D': 4, 'E': 5, 'F': 6,
    'G': 7, 'H': 8, 'I': 9, 'J': 10, 'K': 11, 'L': 12,
    'M': 13, 'N': 14, 'O': 15, 'P': 16, 'Q': 17, 'R': 18,
    'S': 19, 'T': 20, 'U': 21, 'V': 22, 'W': 23, 'X': 24,
    'Y': 25, 'Z': 26
})

print("Configuration:")
print(f"  Device: {DEVICE}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Epochs: {EPOCHS}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Number of classes: {len(ENCODER) + 1}")

# Load data
print("\nLoading data...")
labels = np.load('data/labels.npy', allow_pickle=True)
labels = np.array([ENCODER[x] for x in labels])

imgs = np.load('data/images.npy')
imgs = imgs.astype('float32') / 255.0

# Add channel dimension
if len(imgs.shape) == 3:
    imgs = np.expand_dims(imgs, -1)

print(f"Images shape: {imgs.shape}")
print(f"Labels shape: {labels.shape}")

# Shuffle
labels, imgs = shuffle(labels, imgs, random_state=42)

# Split 75/25
split = int(len(labels) * 0.75)
imgs_train, imgs_test = imgs[:split], imgs[split:]
labels_train, labels_test = labels[:split], labels[split:]

print(f"\nTrain set: {imgs_train.shape[0]} samples")
print(f"Test set: {imgs_test.shape[0]} samples")

## Step 4: Visualize samples

In [ ]:
# Visualize samples
fig, axes = plt.subplots(2, 5, figsize=(12, 6))
fig.suptitle('Sample Images', fontsize=16)

for i, ax in enumerate(axes.flat):
    idx = np.random.randint(0, len(imgs_train))
    letter = ENCODER.inverse[labels_train[idx]]
    ax.imshow(imgs_train[idx, :, :, 0], cmap='gray')
    ax.set_title(f'Letter: {letter}')
    ax.axis('off')

plt.tight_layout()
plt.show()

## Step 5: Convert to PyTorch tensors and create DataLoaders

In [ ]:
# Convert to PyTorch tensors
# PyTorch expects (N, C, H, W) format
imgs_train_tensor = torch.from_numpy(imgs_train).permute(0, 3, 1, 2).float()
labels_train_tensor = torch.from_numpy(labels_train).long()

imgs_test_tensor = torch.from_numpy(imgs_test).permute(0, 3, 1, 2).float()
labels_test_tensor = torch.from_numpy(labels_test).long()

print(f"Train tensor shape: {imgs_train_tensor.shape}")
print(f"Test tensor shape: {imgs_test_tensor.shape}")

# Create datasets and dataloaders
train_dataset = TensorDataset(imgs_train_tensor, labels_train_tensor)
test_dataset = TensorDataset(imgs_test_tensor, labels_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"\nTrain batches: {len(train_loader)}")
print(f"Test batches: {len(test_loader)}")

## Step 6: Define the CNN model

In [ ]:
class AlphabetCNN(nn.Module):
    def __init__(self, num_classes=27):
        super(AlphabetCNN, self).__init__()
        
        # Convolutional layers
        self.conv1 = nn.Conv2d(1, 256, kernel_size=5, padding=0)
        self.pool1 = nn.MaxPool2d(kernel_size=2)
        self.drop1 = nn.Dropout(0.3)
        
        self.conv2 = nn.Conv2d(256, 512, kernel_size=5, padding=0)
        self.pool2 = nn.MaxPool2d(kernel_size=2)
        self.drop2 = nn.Dropout(0.3)
        
        self.conv3 = nn.Conv2d(512, 1024, kernel_size=5, padding=0)
        self.pool3 = nn.MaxPool2d(kernel_size=2)
        self.drop3 = nn.Dropout(0.3)
        
        # Fully connected layers
        self.fc1 = nn.Linear(1024 * 2 * 2, 128)
        self.fc2 = nn.Linear(128, num_classes)
        
        self.relu = nn.ReLU()
    
    def forward(self, x):
        # Conv block 1
        x = self.relu(self.conv1(x))
        x = self.pool1(x)
        x = self.drop1(x)
        
        # Conv block 2
        x = self.relu(self.conv2(x))
        x = self.pool2(x)
        x = self.drop2(x)
        
        # Conv block 3
        x = self.relu(self.conv3(x))
        x = self.pool3(x)
        x = self.drop3(x)
        
        # Flatten
        x = x.view(x.size(0), -1)
        
        # Fully connected
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        
        return x

# Create model
model = AlphabetCNN(num_classes=len(ENCODER) + 1)
model = model.to(DEVICE)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"Model created with {total_params:,} parameters")
print(f"\n{model}")

## Step 7: Define loss function and optimizer

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(f"Loss function: {criterion}")
print(f"Optimizer: {optimizer}")

## Step 8: Train the model

In [ ]:
def train_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for imgs, labels in train_loader:
        imgs = imgs.to(device)
        labels = labels.to(device)
        
        # Forward pass
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Statistics
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
    
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = correct / total
    
    return epoch_loss, epoch_acc

def validate(model, test_loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs = imgs.to(device)
            labels = labels.to(device)
            
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    
    epoch_loss = running_loss / len(test_loader)
    epoch_acc = correct / total
    
    return epoch_loss, epoch_acc

# Training loop
print(f"Training for {EPOCHS} epochs...\n")

train_losses = []
val_losses = []
train_accs = []
val_accs = []
best_val_acc = 0
patience = 2
patience_counter = 0

for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, DEVICE)
    val_loss, val_acc = validate(model, test_loader, criterion, DEVICE)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)
    
    print(f"Epoch {epoch+1:2d}/{EPOCHS} | "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    
    # Early stopping
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        # Save best model
        torch.save(model.state_dict(), 'best_model.pth')
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"\nEarly stopping at epoch {epoch+1}")
            # Load best model
            model.load_state_dict(torch.load('best_model.pth'))
            break

print("\n✓ Training complete!")

## Step 9: Evaluate on test set

In [ ]:
test_loss, test_acc = validate(model, test_loader, criterion, DEVICE)
print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")

## Step 10: Plot training history

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
ax1.plot(train_accs, label='Train Accuracy', linewidth=2)
ax1.plot(val_accs, label='Val Accuracy', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.set_title('Model Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Loss
ax2.plot(train_losses, label='Train Loss', linewidth=2)
ax2.plot(val_losses, label='Val Loss', linewidth=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.set_title('Model Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 11: Save the model

In [ ]:
os.makedirs('models', exist_ok=True)

# Save model state dict
model_path = 'models/letter.pth'
torch.save(model.state_dict(), model_path)
print(f"✓ Model saved to {model_path}")

# Save encoder
import json
encoder_path = 'models/encoder.json'
encoder_dict = {str(k): int(v) for k, v in ENCODER.items()}
with open(encoder_path, 'w') as f:
    json.dump(encoder_dict, f)
print(f"✓ Encoder saved to {encoder_path}")

## Step 12: Download files

In [ ]:
files.download('models/letter.pth')
print("✓ Downloaded letter.pth")

files.download('models/encoder.json')
print("✓ Downloaded encoder.json")

print("\n" + "="*50)
print("Training Complete!")
print("="*50)
print(f"\n✓ Model trained with {test_acc*100:.2f}% accuracy")
print(f"\nDownload the files and use them with PyTorch Flask app")